# 第 1 周 Day 1：VLA 闭环与四类变量 — Notebook 作业

[← Notebook 总览](../README.md) · [本周 Notebook](README.md) · [课程正文](../../week-01/day-01.md) · [Goal 进度](../../PROGRESS.md) · [Week 01 / Day 02 →](day-02.ipynb)

> 状态：**未提交**。直接编辑各个 Markdown/Code 单元格；“教练验收区”不要预填。


## Goal

预计 60–90 分钟。完成后能用自己的话区分 VLM 与 VLA，写出策略函数 $a_t = \pi_\theta(o_t, l, s_t)$，并画出包含环境反馈的闭环，而不只是一次模型前向。

### 我的目标复述

observation 是外部世界观测，language 是任务指令，proprioception 是机器人状态，action 是执行器动作。


## Setup

| 字段 | 我的记录 |
|---|---|
| 实际投入时间 | 【填写】 |
| 完成日期 | 【填写】 |
| Python / PyTorch | 【填写；纯理论日写“不适用”】 |
| CPU / GPU / 仿真器 | 【填写】 |
| 资源等级 | 【L0 / L1 / L2】 |
| 产物路径 | 【填写】 |

生成时环境检查（2026-09-01）：当前可见 Python 未检测到 Jupyter、ipykernel、nbformat、PyTorch 或 NumPy。本 Notebook 已做结构验证，但在该环境中尚未执行。


In [ ]:
# 可选：Notebook 环境可用后运行此单元，记录基础环境。
import platform
import sys

print("python:", sys.version)
print("platform:", platform.platform())


## Context：知识点及其在 VLA 中的作用

- `observation o_t`：外部世界的观测，常见为 RGB/深度图；
- `language l`：任务条件，例如“把红色方块放进蓝色盒”；
- `proprioception s_t`：机器人自身状态，例如关节角、末端位姿和夹爪状态；
- `action a_t`：控制器可执行的数值动作；
- offline training 与 closed-loop rollout：前者在固定数据上拟合，后者让预测改变未来输入。

这组变量是后续数据集、模型和评估的共同接口。


## Concepts：概念、公式、形状与数据流

策略写作：

$$
a_t=\pi_\theta(o_t,l,s_t)
$$

以批大小 `B=4`、RGB 为 `224×224`、文本长度 `L=16`、状态 8 维、动作 7 维为例：

| 变量 | 语义 | 形状 |
|---|---|---|
| $o_t$ | 当前 RGB | `[4, 3, 224, 224]` |
| $l$ | 尚未 embedding 的 token id | `[4, 16]` |
| $s_t$ | 位置 3 + 四元数 4 + 夹爪 1 | `[4, 8]` |
| $a_t$ | 位移 3 + 旋转增量 3 + 夹爪 1 | `[4, 7]` |

闭环数据流：

```text
(o_t, l, s_t) -> policy -> a_t -> environment
     ^                                |
     +-------- (o_{t+1}, s_{t+1}) <---+
```

语言通常在一个 episode 内不变，观测、状态和动作随时间变化。动作的单位与坐标系今天先记录为“必须声明”，第 2 周再系统处理。


## Learning Steps

1. 用 10 分钟写出 VLM 与 VLA 各一个输入输出例子。
2. 用 15 分钟把“红方块放入蓝盒”映射到四类变量。
3. 用 20 分钟画闭环，标注 `t -> t+1`。
4. 用 15 分钟解释离线动作误差如何在 rollout 中积累。
5. 用剩余时间完成作业并自查形状。


## Steps：必做作业

### 课程题目

1. 写一段不超过 150 字的 VLA 定义，并指出 VLM 与 VLA 的核心差别。
2. 为“把红色方块放进蓝色盒子”画完整闭环图。
3. 按上表设定，写出四个输入输出张量的形状和 dtype；token id 应为整数，网络输入与动作通常为浮点数。
4. 分析两种失败：视觉定位偏 2 cm；夹爪命令语义反了。说明它们如何影响下一时刻。

下面每道题都有独立作答单元。文字、表格、公式或 Mermaid 写在 Markdown 单元；可运行代码写在后面的 Code 单元。


### 第 1 题作答

**我的回答：** 通过一个策略函数，输入为环境观测，机器人状态和任务指令，输出执行器动作的模型叫做VLA


### 第 2 题作答

**我的闭环图：**

```text
语言指令 l：“把红色方块放进蓝色盒子”（episode 内保持不变）

  (o_t：红块和蓝盒的相机图像,
   s_t：末端位姿、关节角、夹爪状态,
   l)
            |
            v
        VLA policy
            |
            v
  a_t = [Δx, Δy, Δz, Δroll, Δpitch, Δyaw, gripper]
            |
            v
  环境执行动作：靠近红块 → 抓取 → 移向蓝盒 → 松爪
            |
            v
  产生新的观测 o_{t+1} 和机器人状态 s_{t+1}
            |
            +--------反馈给 VLA，继续下一步--------+
                           |
              直到红块进入蓝盒或任务终止

  注意 a_t 应是机器人可以执行的数值动作，不能直接写成自然语言“抓起红色方块”。
```


### 第 3 题作答

| 变量 | 含义 | shape | dtype |
|---|---|---|---|
| $o_t$ | 当前 RGB 图像观测 | `[4, 3, 224, 224]` | `torch.float32` |
| $l$ | 尚未 embedding 的文本 token id | `[4, 16]` | `torch.int64`（`torch.long`） |
| $s_t$ | 机器人状态：位置 3 + 四元数 4 + 夹爪 1 | `[4, 8]` | `torch.float32` |
| $a_t$ | 动作：位移 3 + 旋转增量 3 + 夹爪 1 | `[4, 7]` | `torch.float32` |

设批大小为 $B$，图像通道数为 $C$，图像高宽为 $H, W$，文本长度为 $L$，状态维度为 $d_s$，动作维度为 $d_a$。本题中：

$B=4,\; C=3,\; H=W=224,\; L=16,\; d_s=8,\; d_a=7$
| 变量 | 符号化 shape | 本题具体 shape | dtype |
|---|---|---|---|
| $o_t$（RGB 观测） | `[B, C, H, W]` | `[4, 3, 224, 224]` | `torch.float32` |
| $l$（token id） | `[B, L]` | `[4, 16]` | `torch.int64` / `torch.long` |
| $s_t$（机器人状态） | `[B, d_s]` | `[4, 8]` | `torch.float32` |
| $a_t$（动作输出） | `[B, d_a]` | `[4, 7]` | `torch.float32` |

其中策略接口为：
$$
a_t = \pi_\theta(o_t, l, s_t).
$$

其中，$l$ 是离散的词表编号，因此必须用整数类型；$o_t$ 、$s_t$ 和 $a_t$ 是送入网络或由网络预测的连续数值，通常使用 float32。图像这里指预处理后送入网络的浮点张量。

### 第 4 题作答

失败一：视觉定位偏差 2 cm。
相机把红色方块的位置估计错了 2 cm，策略根据错误的观测 $o_t$ 计算动作 $a_t$，会让机械臂朝错误位置移动。环境实际执行后，机械臂末端的位置变成 $s_{t+1}$，而下一帧图像 $o_{t+1}$ 会显示夹爪没有对准方块，可能抓空、碰到方块或把方块推偏。这样下一步策略接收到的状态已偏离原本示范轨迹；若不能及时纠正，位置误差会在闭环中继续累积。
定位偏差 → 错误动作 $a_t$ → 机械臂未对准 → $o_{t+1}$, $s_{t+1}$ 偏离预期 → 下一步更难纠正

失败二：夹爪命令语义反了。
假设策略输出“闭合夹爪”以抓取方块，但控制器把该命令解释成“打开夹爪”。环境实际执行的是相反动作，因此在 $t+1$ 时夹爪没有抓住方块，图像 $o_{t+1}$ 中方块仍留在原处，机器人状态 $s_{t+1}$ 中的夹爪开合状态也与策略预期相反。之后策略会基于这个意外状态继续预测动作，可能反复抓空，或者在本应放下方块时没有释放。
策略想闭合 → 控制器实际打开 → 未抓住方块 → $o_{t+1}$, $s_{t+1}$ 与预期不一致 → 后续动作继续出错


核心记住一句：$a_t$ 执行后会改变环境，因此会产生新的 $o_{t+1}$ 和 $s_{t+1}$。

### 可运行代码 / 实验区

纯理论日可以保留为空；代码日请将实现拆成短小单元，并保留关键输出。


In [ ]:
# 在此编写或运行当天代码。
# 建议先写清输入 shape、dtype、设备和随机种子。


## Checks：输入、预期输出与验证

- 输入：任务描述、`B=4,H=W=224,L=16,d_s=8,d_a=7`。
- 预期输出：一张含环境反馈的图、一张完整张量表、两条因果失败链。
- 验证：闭环必须出现 `o_{t+1}` 和 `s_{t+1}`；四个 batch 维都为 4；`o_t` 通道在第二维；动作不是自然语言。

### 我的验证记录

| 检查项 | 实际结果 | 是否符合 | 证据 |
|---|---|---|---|
| 输入 shape / schema | 【填写】 | 【填写】 | 【填写】 |
| 输出 shape / schema | 【填写】 | 【填写】 | 【填写】 |
| dtype、范围和单位 | 【填写】 | 【填写】 | 【填写】 |
| 正向测试 | 【填写】 | 【填写】 | 【填写】 |
| 负向测试 / 错误注入 | 【填写】 | 【填写】 | 【填写】 |
| 指标分子 / 分母 / seed | 【填写】 | 【填写】 | 【填写】 |

> 尚未运行的内容必须标为“预期结果”，不能作为实际证据。


In [ ]:
# 在此编写 shape、dtype、数值范围、断言或负向测试。


## Evidence：提交与复现证据

提交 Markdown，固定包含：`一句话定义`、`闭环图`、`变量与形状表`、`两个失败链`、`仍不确定的点`、`投入分钟数`。闭环图可用 Mermaid、ASCII 或清晰手绘截图。

### 我的证据

- 代码路径：【填写】
- 配置路径：【填写】
- 数据 / checkpoint / commit 或哈希：【填写】
- 实际命令：【填写】
- 退出码：【填写】
- 关键输出：【填写】
- 结果说明了什么：【填写】
- 结果没有说明什么：【填写】
- 失败现象与定位证据：【填写】

### VLA 约束

| 约束 | 我的定义 |
|---|---|
| 图像布局、颜色顺序和范围 | 【填写】 |
| 文本 token、padding 与 mask | 【填写】 |
| 机器人状态各维含义 | 【填写】 |
| 坐标系、长度和角度单位 | 【填写】 |
| 动作空间及逐维定义 | 【填写】 |
| observation/action 时间对齐 | 【填写】 |
| 控制频率 / action chunk | 【填写】 |
| 归一化及统计量来源 | 【填写】 |
| 随机种子与数据划分 | 【填写】 |


## Self-check：课程自测

1. 相机图像与末端位姿为什么不是同一种状态？
2. 哪个量的执行会改变下一帧输入？
3. 指令在 episode 中不变是否意味着模型每步都不需要它？为什么？
4. 训练集动作 MSE 很低时，仍需报告什么闭环指标？


### 自测第 1 题作答

【双击此 Markdown 单元格，在这里填写答案。】


### 自测第 2 题作答

【双击此 Markdown 单元格，在这里填写答案。】


### 自测第 3 题作答

【双击此 Markdown 单元格，在这里填写答案。】


### 自测第 4 题作答

【双击此 Markdown 单元格，在这里填写答案。】


## Help：排查、最低完成线与提高

### 常见错误

- 把“目标位置”直接当机器人状态：先问它来自外部相机还是机器人传感器。
- 写成 `[B,H,W,C]`：PyTorch 常用 NCHW，此课按 `[B,C,H,W]`。
- 只画模型不画环境：检查动作之后是否产生下一观测。
- 认为低 loss 等于成功：检查 rollout 输入是否已偏离示范分布。

### 最低完成线

完成闭环图、四变量表，以及一句“为什么低 loss 不保证成功”的解释，约 30 分钟。

### 可选提高

给同一任务分别设计单步动作与 4 步 action chunk 的输出形状，并列出各自风险；暂不实现。


## Rubric

总分 100，80 分通过：定义与 VLM/VLA 区分 20；闭环完整 25；四变量语义 20；形状与 dtype 20；失败因果链 15。若闭环没有反馈、混淆 `o_t/s_t` 或将动作写成文本，即使总分达到 80 也不通过。

### 提交前检查

- [ ] 已逐项完成必做作业；
- [ ] 已区分实际结果与预期结果；
- [ ] 已保留代码输出、日志、表格或推理证据；
- [ ] 已记录适用的 shape、坐标系、单位、动作和时间约定；
- [ ] 已完成验证或明确写出无法执行的原因；
- [ ] 已回答全部自测题；
- [ ] 已记录仍不确定的点或失败案例。


## Coach Review（学习者请勿填写）

| 字段 | 验收结果 |
|---|---|
| 证据完整性 | 待验收 |
| Rubric 得分 | /100 |
| 门槛项 | 待验收 |
| 当天状态 | 未提交 |
| 具体缺口 |  |
| 最小补救任务 |  |
| 复验结果 |  |
| 下一课程 |  |


## Next Steps

完成后保存 Notebook，并把路径发到学习对话：

`docs/vla-learning/notebooks/week-01/day-01.ipynb`

教练验收通过后才会更新 `PROGRESS.md`。

[← Notebook 总览](../README.md) · [本周 Notebook](README.md) · [课程正文](../../week-01/day-01.md) · [Week 01 / Day 02 →](day-02.ipynb)
